# ST Guitar — Stage 7G-E3-R2 Colab Learning Demonstration

Bu notebook'un amacı **modelin gerçekten öğrenmesini görünür hale getirmektir**.

Göreceğin ana değerler:
- Train Loss
- Val Loss
- Validation Macro-F1
- Balanced Accuracy
- COMPACT Precision / Recall
- TP / FP / FN / TN
- Always-OPEN_LOW baseline ve modele göre kazanç
- Epoch → Loss ve Epoch → Macro-F1 grafikleri

**TRAIN hücresini yalnız sen manuel çalıştıracaksın.**

Bilimsel sınır:
- Eğitimde yalnız eski E3 development Teacher-GOLD: 399 decisive row.
- 240 E3-E cevapları bu notebook'a yüklenmez ve eğitimde kullanılmaz.
- Stage7E kullanılmaz.
- Checkpoint/model dosyası kaydedilmez.
- Production entegrasyonu yapılmaz.
- `LocF1@2px` bu model için uygulanamaz; model piksel konumu değil OPEN_LOW/COMPACT tercihi tahmin eder.


In [ ]:
PINNED_CODE_SHA = "3fa43bf0ed138fe50c0f3374b3bdfa6f5768b903"
ANIMETAB_COMMIT = "18c0993cbe0a0948cbf0b7768bcb09ff81c23a9a"
EXPECTED_CHOICES_SHA256 = "db0e752ec7b9e0e1b333a217d904175f4e57cd89a32b2511330ebab7b8c6c12e"
assert len(PINNED_CODE_SHA) == 40
assert len(ANIMETAB_COMMIT) == 40


In [ ]:
!rm -rf st-guitar-fingering-training
!git clone -q https://github.com/khfy7wpr5p-maker/st-guitar-fingering-training.git
%cd st-guitar-fingering-training
!git checkout -q $PINNED_CODE_SHA
!pip -q install -e .


## Tek dosya yükle

Yalnız şu eski development Teacher-GOLD dosyasını yükle:

`ST_Guitar_E3_Batch01_choices_400of400.json`

**E3-E 240 cevap dosyasını yükleme.**


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
# PRE-FIT DATA RECONSTRUCTION + IDENTITY CHECK — BU HÜCRE MODEL EĞİTMEZ
from __future__ import annotations
import hashlib, json, platform, subprocess, urllib.parse, urllib.request
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
import sklearn

from st_guitar_fingering_training.target_free_musicxml import parse_target_free_musicxml
from st_guitar_fingering_training.stage7g_e3_e_a3 import reconstruct_frozen_open_low_compact_specialists
from st_guitar_fingering_training.stage7g_e3_r2_learning import (
    STAGE7G_E3_R2_CONFIG,
    build_stage7g_e3_r2_disagreement_pool,
    rows_from_choices,
)

repo_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert repo_sha == PINNED_CODE_SHA, (repo_sha, PINNED_CODE_SHA)

choices_path = Path("ST_Guitar_E3_Batch01_choices_400of400.json")
assert choices_path.is_file(), "STOP: 400-of-400 choices JSON bulunamadı"
actual_choices_sha = hashlib.sha256(choices_path.read_bytes()).hexdigest()
assert actual_choices_sha == EXPECTED_CHOICES_SHA256, (
    "STOP: choices SHA mismatch", actual_choices_sha, EXPECTED_CHOICES_SHA256
)

manifest_path = Path("evidence/stage7g_c_r1_animetab_batch01_manifest.json")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest["family_count"] == 40
assert manifest["part_id"] == "P1"
assert manifest["staff_id"] == "2"
assert manifest["pitch_mode"] == "sounding_exact"
assert manifest["tuning_midi"] == [64, 59, 55, 50, 45, 40]

sources = []
with TemporaryDirectory() as tmp:
    tmp_root = Path(tmp)
    for index, item in enumerate(manifest["sources"], start=1):
        filename = item["filename"]
        encoded = urllib.parse.quote(filename, safe="")
        url = (
            "https://raw.githubusercontent.com/amamiya-yuuko/AnimeTAB/"
            + ANIMETAB_COMMIT
            + "/AnimeTAB/Entire%20songs/"
            + encoded
        )
        req = urllib.request.Request(url, headers={"User-Agent": "st-guitar-e3-r2-colab-v1"})
        with urllib.request.urlopen(req, timeout=45) as response:
            data = response.read()
        digest = hashlib.sha256(data).hexdigest()
        assert digest == item["sha256"], f"STOP: source SHA drift: {filename}"
        local = tmp_root / f"{index:03d}.xml"
        local.write_bytes(data)
        sources.append(
            parse_target_free_musicxml(
                local,
                family_id=item["family_id"],
                tuning=manifest["tuning_midi"],
                pitch_mode=manifest["pitch_mode"],
                part_id=manifest["part_id"],
                staff_id=manifest["staff_id"],
            )
        )

models, specialist_guard = reconstruct_frozen_open_low_compact_specialists()
assert specialist_guard["status"] == "PASS_STAGE7B_C2_OPEN_LOW_COMPACT_RECONSTRUCTION"

pool = build_stage7g_e3_r2_disagreement_pool(
    tuple(sources),
    specialist_models=models,
)
choices = json.loads(choices_path.read_text(encoding="utf-8"))
rows, preflight = rows_from_choices(pool, choices)
assert preflight["status"] == "R2_PREFLIGHT_PASS_STOP_BEFORE_MANUAL_TRAIN"

identity = {
    "repository": "khfy7wpr5p-maker/st-guitar-fingering-training",
    "code_sha": repo_sha,
    "animetab_commit": ANIMETAB_COMMIT,
    "choices_sha256": actual_choices_sha,
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "r2_config": STAGE7G_E3_R2_CONFIG,
    "e3e_teacher_gold_used": False,
    "stage7e_used": False,
    "checkpoint_retained": False,
    "production_integration": False,
}
print(json.dumps(identity, indent=2))
print(json.dumps(preflight, indent=2))
print("\n===== STOP =====")
print("PREFLIGHT PASS. 399 decisive Teacher-GOLD row hazır.")
print("Henüz model eğitilmedi.")
print("Şimdi aşağıdaki MANUAL TRAIN hücresini SEN çalıştır.")
PREFLIGHT_READY = True


# ▶ MANUAL TRAIN

Bu hücre **gerçek R2 eğitimidir**.

Çalıştırdığında 60 epoch boyunca model 399 development Teacher-GOLD satırının family-isolated train kısmından öğrenir.

Çalışma bittiğinde her epoch için `Train Loss`, `Val Loss`, `Macro-F1`, `Precision`, `Recall` değerlerini göreceksin.


In [ ]:
# MANUAL TRAIN CELL — BU HÜCREYİ KULLANICI ÇALIŞTIRIR
assert PREFLIGHT_READY is True

from st_guitar_fingering_training.stage7g_e3_r2_learning import stage7g_e3_r2_learning_report

report = stage7g_e3_r2_learning_report(rows)

print(
    f"{'Epoch':>5}  {'TrainLoss':>10}  {'ValLoss':>10}  "
    f"{'ValMacroF1':>10}  {'BalAcc':>8}  {'C-Prec':>8}  {'C-Rec':>8}"
)
print("-" * 76)
for h in report["history"]:
    print(
        f"{h['epoch']:5d}  "
        f"{h['train_loss']:10.4f}  "
        f"{h['val_loss']:10.4f}  "
        f"{h['val_macro_f1']:10.4f}  "
        f"{h['val_balanced_accuracy']:8.4f}  "
        f"{h['val_compact_precision']:8.4f}  "
        f"{h['val_compact_recall']:8.4f}"
    )

TRAIN_COMPLETE = True
print("\n===== TRAINING COMPLETE =====")


In [ ]:
# ÖĞRENME GRAFİĞİ 1 — LOSS
assert TRAIN_COMPLETE is True
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in report["history"]]
train_loss = [h["train_loss"] for h in report["history"]]
val_loss = [h["val_loss"] for h in report["history"]]

plt.figure(figsize=(9, 5))
plt.plot(epochs, train_loss, label="Train Loss")
plt.plot(epochs, val_loss, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Log Loss")
plt.title("ST Guitar R2 — Train / Validation Loss")
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()


In [ ]:
# ÖĞRENME GRAFİĞİ 2 — VALIDATION MACRO-F1
assert TRAIN_COMPLETE is True
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in report["history"]]
macro_f1 = [h["val_macro_f1"] for h in report["history"]]

plt.figure(figsize=(9, 5))
plt.plot(epochs, macro_f1, label="Validation Macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("ST Guitar R2 — Validation Macro-F1")
plt.ylim(0.0, 1.0)
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()


In [ ]:
# SON ÖĞRENME ÖZETİ
assert TRAIN_COMPLETE is True

first = report["history"][0]
last = report["history"][-1]
base = report["baseline"]
final = report["final_validation"]

print("===== LEARNING SUMMARY =====")
print(f"Epoch-1 Val Loss             : {first['val_loss']:.4f}")
print(f"Epoch-60 Val Loss            : {last['val_loss']:.4f}")
print(f"Val Loss change              : {final['val_loss_change_epoch1_to_final']:+.4f}  (negative = lower loss)")
print()
print(f"Epoch-1 Val Macro-F1         : {first['val_macro_f1']:.4f}")
print(f"Epoch-60 Val Macro-F1        : {last['val_macro_f1']:.4f}")
print(f"Always-OPEN_LOW Macro-F1     : {base['always_open_low_macro_f1']:.4f}")
print(f"Macro-F1 gain vs baseline    : {final['macro_f1_gain_vs_always_open_low']:+.4f}")
print()
print(f"Validation Accuracy          : {final['accuracy']:.4f}")
print(f"Always-OPEN_LOW Accuracy     : {base['always_open_low_accuracy']:.4f}")
print(f"Accuracy gain                : {final['accuracy_gain_vs_always_open_low']:+.4f}")
print(f"Balanced Accuracy            : {final['balanced_accuracy']:.4f}")
print(f"COMPACT Precision            : {final['compact_precision']:.4f}")
print(f"COMPACT Recall               : {final['compact_recall']:.4f}")
print(f"TP / FP / FN / TN            : {final['tp']} / {final['fp']} / {final['fn']} / {final['tn']}")
print()
print("LocF1@2px                    : N/A — bu model piksel konumu tahmin etmiyor")
print("Checkpoint retained          : False")
print("E3-E Teacher-GOLD used       : False")


In [ ]:
# AGGREGATE SONUÇ DOSYASINI DIŞA AKTAR — MODEL/CHECKPOINT YOK
assert TRAIN_COMPLETE is True
from google.colab import files
from pathlib import Path
import json

result = {
    "schema": "st-guitar-stage7g-e3-r2-colab-learning-result-v1",
    "identity": identity,
    "preflight": preflight,
    "report": report,
    "checkpoint_retained": False,
    "e3e_teacher_gold_used": False,
    "stage7e_used": False,
    "production_integration": False,
}
output = Path("ST_Guitar_Stage7G_E3_R2_Colab_Learning_result.json")
output.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
files.download(str(output))
print("Exported:", output)
